In [ ]:
# Ячейка 1 — конфиг, загрузка и GOLD check

from pathlib import Path
import os, re, json, random, math, time, warnings
import unicodedata

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

SEEDS = [42, 1337]   # два seed, чтобы не раздувать время
BASE_SEED = 42

BASE = Path("/kaggle/input")
COMP = None
for p in BASE.rglob("sample_submission.csv"):
    COMP = p.parent
    break
if COMP is None:
    raise FileNotFoundError("Не нашёл sample_submission.csv. Проверь Add Data.")

train = pd.read_csv(COMP / "train.csv")
test = pd.read_csv(COMP / "test.csv")
train_series = pd.read_csv(COMP / "train_series.csv")
test_series = pd.read_csv(COMP / "test_series.csv")
sample_sub = pd.read_csv(COMP / "sample_submission.csv")

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

GOLD = train[train[LABELS].notna().all(axis=1)].copy()
print("GOLD fully annotated studies:", len(GOLD), flush=True)
if len(GOLD) == 0:
    raise RuntimeError("Не нашёл gold-строки в train.csv. Ожидались ~58 fully annotated.")
print("GOLD positive rate:", flush=True)
print(GOLD[LABELS].mean().round(3), flush=True)

# Конфиг: назад к живому v0.3/v0.6, но с gold weight.
IMG_SIZE = 320
SLICES_PER_SERIES = 10
N_TRAIN_STUDIES = 512
EPOCHS = 2
BATCH_SIZE = 8
LR = 3e-4
NUM_WORKERS = 0
AMP = True
GRAD_CLIP = 1.0
VAL_FRAC = 0.20
GOLD_WEIGHT = 3.0

print("CONFIG v0.7:", {
    "SEEDS": SEEDS, "IMG_SIZE": IMG_SIZE, "SLICES_PER_SERIES": SLICES_PER_SERIES,
    "N_TRAIN_STUDIES": N_TRAIN_STUDIES, "EPOCHS": EPOCHS, "BATCH_SIZE": BATCH_SIZE,
    "LR": LR, "VAL_FRAC": VAL_FRAC, "GOLD_WEIGHT": GOLD_WEIGHT,
}, flush=True)

In [ ]:
# Ячейка 2 — extractor v2 + agreement на 58 GOLD

_PRE = str.maketrans({"ı": "i", "İ": "i", "I": "i", "ß": "ss", "đ": "d", "Đ": "d", "ø": "o", "Ø": "o", "æ": "ae", "Æ": "ae"})

def normalize(text):
    if not isinstance(text, str):
        return ""
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("\xad", " ")
    text = re.sub(r"[_\-/\\]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

NEG_RE = re.compile(
    r"(\bno\b|\bnot\b|\bwithout\b|\bsin\b|\bno hay\b|\bkein\b|\bkeine\b|\baucun\b|\baucune\b|"
    r"\bgeen\b|\bniet\b|\bsem\b|\bnon\b|\bнет\b|\bбез\b|\bintact\b|\bnormal\b|\bwithin normal\b|"
    r"\bunremarkable\b|\bconservad)",
    re.I
)

RULES = {
    "ACL": [r"\bacl\b", r"\blca\b", r"cruzado anterior", r"croise anterieur", r"vorderes kreuzband", r"крестообраз"],
    "MCL": [r"\bmcl\b", r"colateral medial", r"collateral medial", r"mediales kollateral", r"медиальн.{0,20}коллатерал"],
    "Medial Meniscus": [r"medial meniscus", r"meniscus medial", r"menisco medial", r"menisco interno", r"innenmeniskus", r"медиальн.{0,20}мениск"],
    "Lateral Meniscus": [r"lateral meniscus", r"meniscus lateral", r"menisco lateral", r"menisco externo", r"aussenmeniskus", r"латеральн.{0,20}мениск"],
    "Medial OA": [r"medial.{0,40}(osteoarth|arthrosis|arthrose)", r"(osteoarth|arthrosis|arthrose).{0,40}medial", r"artrosis.{0,40}medial", r"femorotibial medial", r"медиальн.{0,40}остеоартр"],
    "Lateral OA": [r"lateral.{0,40}(osteoarth|arthrosis|arthrose)", r"(osteoarth|arthrosis|arthrose).{0,40}lateral", r"artrosis.{0,40}lateral", r"femorotibial lateral", r"латеральн.{0,40}остеоартр"],
    "PF OA": [r"patellofemoral.{0,40}(osteoarth|arthrosis|arthrose|chondrop|chondros)", r"(osteoarth|arthrosis|arthrose).{0,40}patellofemoral", r"femoropatelar", r"retropatellar", r"пателлофеморал"],
    "Effusion": [r"effusion", r"derrame", r"erguss", r"epanchement", r"versamento", r"joint fluid", r"выпот"],
    "Synovitis": [r"\bsynovitis\b", r"\bsinovitis\b", r"синовит"],
    "Baker's": [r"baker", r"popliteal cyst", r"quiste popl", r"poplitea", r"беккер", r"бейкер"],
    "Contusion": [r"contusion", r"bone bruise", r"bone marrow edema", r"edema oseo", r"костномозгов"],
    "Fracture": [r"fracture", r"fractura", r"fraktur", r"перелом"],
}

def clauses(text):
    t = normalize(text)
    return [s.strip() for s in re.split(r"[\.\!\?\n\r]+", t) if s.strip()]

def weak_one(text):
    hits = {lab: {"pos": 0, "neg": 0} for lab in LABELS}
    for s in clauses(text):
        neg = bool(NEG_RE.search(s))
        for lab in LABELS:
            if any(re.search(p, s, flags=re.I) for p in RULES[lab]):
                hits[lab]["neg" if neg else "pos"] += 1
    return {lab: (1.0 if hits[lab]["pos"] > 0 else (0.0 if hits[lab]["neg"] > 0 else np.nan)) for lab in LABELS}

cache = Path("/kaggle/working/train_weak_v2_gold.csv")
if cache.exists():
    weak = pd.read_csv(cache)
    print("loaded weak cache:", weak.shape, flush=True)
else:
    print("extract weak labels...", flush=True)
    t0 = time.time()
    weak = pd.DataFrame([weak_one(x) for x in train["Report"].fillna("")])
    weak.insert(0, "StudyInstanceUID", train["StudyInstanceUID"].values)
    weak.to_csv(cache, index=False)
    print("weak done", time.time() - t0, "s", flush=True)

# Agreement на GOLD
from sklearn.metrics import roc_auc_score
wp = weak.set_index("StudyInstanceUID").loc[GOLD["StudyInstanceUID"], LABELS]
rows = []
for c in LABELS:
    y = GOLD[c].values.astype(int)
    p = wp[c].values
    m = ~np.isnan(p)
    sil = float(np.isnan(p).mean())
    auc = np.nan
    if m.sum() > 0 and len(np.unique(y[m])) == 2:
        try:
            auc = roc_auc_score(y[m], p[m])
        except Exception:
            pass
    rows.append({"label": c, "auc_on_gold": auc, "silence": sil, "gold_pos": int(y.sum())})
rep = pd.DataFrame(rows)
print(rep.round(4).to_string(index=False), flush=True)
print("macro agreement AUC:", float(np.nanmean(rep["auc_on_gold"])), flush=True)
print("mean silence:", float(rep["silence"].mean()), flush=True)

In [ ]:
# Ячейка 3 — выбор серий и sample weights с GOLD_WEIGHT

PLANES = ["Axial", "Sagittal", "Coronal"]

def select_series(sdf, split, desc=""):
    rows = []
    studies = list(sdf.groupby("StudyInstanceUID"))
    t0 = time.time()
    for i, (study, x) in enumerate(studies):
        rec = {"study": study, "split": split}
        for plane in PLANES:
            xp = x[x["Anatomical_Plane"] == plane].copy()
            if len(xp) == 0:
                rec[plane] = ""
                continue
            xp["score"] = xp["Fluid_Sensitive"].fillna(0) * 2 + xp["Fat_Suppression"].fillna(0)
            r = xp.sort_values("score", ascending=False).iloc[0]
            rec[plane] = str(COMP / f"{split}_series" / str(study) / str(r["SeriesInstanceUID"]))
        rows.append(rec)
        if desc and (i % 1500 == 0 or i == len(studies) - 1):
            print(f"[{desc}] {i + 1}/{len(studies)} elapsed={time.time() - t0:.1f}s", flush=True)
    return pd.DataFrame(rows)

train_sel = select_series(train_series, "train", "select train")
test_sel = select_series(test_series, "test", "select test")

train_df = train_sel.merge(weak, left_on="study", right_on="StudyInstanceUID", how="left").drop(columns=["StudyInstanceUID"])
gold_ids = set(GOLD["StudyInstanceUID"])
train_df["sample_w"] = np.where(train_df["study"].isin(gold_ids), GOLD_WEIGHT, 1.0).astype(np.float32)
test_df = test_sel.copy()

print("train_df:", train_df.shape, "test_df:", test_df.shape, flush=True)
print("gold rows in train_df:", int((train_df["sample_w"] > 1).sum()), flush=True)
train_df.to_csv("/kaggle/working/train_selected_series_v0_7.csv", index=False)
test_df.to_csv("/kaggle/working/test_selected_series_v0_7.csv", index=False)

In [ ]:
# Ячейка 4 — dataset/model с sample_w, self-contained fallback

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from PIL import Image
import pydicom

if "IMG_SIZE" not in globals(): IMG_SIZE = 320
if "SLICES_PER_SERIES" not in globals(): SLICES_PER_SERIES = 10
if "AMP" not in globals(): AMP = True
if "LABELS" not in globals():
    LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
if "PLANES" not in globals(): PLANES = ["Axial", "Sagittal", "Coronal"]
if "COMP" not in globals():
    BASE = Path("/kaggle/input")
    COMP = None
    for p in BASE.rglob("sample_submission.csv"):
        COMP = p.parent
        break
    if COMP is None:
        raise FileNotFoundError("Не нашёл sample_submission.csv. Проверь Add Data.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = bool(AMP and DEVICE == "cuda")
torch.backends.cudnn.benchmark = True
print("DEVICE:", DEVICE, "USE_AMP:", USE_AMP, flush=True)

try:
    from pydicom.pixel_data_handlers.util import apply_voi_lut
except Exception:
    apply_voi_lut = None
RESAMPLE = getattr(getattr(Image, "Resampling", Image), "BILINEAR")

def read_dicom(path):
    try:
        ds = pydicom.dcmread(str(path))
        arr = ds.pixel_array
        if apply_voi_lut is not None:
            try: arr = apply_voi_lut(arr, ds)
            except Exception: pass
        arr = arr.astype(np.float32)
        if arr.ndim == 3:
            arr = arr[..., 0] if arr.shape[-1] in [3, 4] else arr[0]
        if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
            arr = arr.max() - arr
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
        h, w = arr.shape
        m = max(h, w)
        canvas = np.zeros((m, m), dtype=np.float32)
        y0 = (m - h) // 2; x0 = (m - w) // 2
        canvas[y0:y0+h, x0:x0+w] = arr
        img = Image.fromarray((canvas * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), RESAMPLE)
        return np.asarray(img).astype(np.float32) / 255.0
    except Exception:
        return np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)

def series_tensor(path):
    zero = torch.zeros((SLICES_PER_SERIES, 1, IMG_SIZE, IMG_SIZE), dtype=torch.float32)
    if not path: return zero
    files = sorted(Path(path).glob("*.dcm"))
    if len(files) == 0: return zero
    idx = np.linspace(0, len(files) - 1, SLICES_PER_SERIES).round().astype(int)
    arr = np.stack([read_dicom(files[i]) for i in idx], axis=0)
    arr = (arr - 0.5) / 0.5
    return torch.from_numpy(arr).float().unsqueeze(1)

class KneeStudyDS(Dataset):
    def __init__(self, df, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.has_labels = has_labels
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        xs, pm = [], []
        for plane in PLANES:
            path = r.get(plane, "")
            if isinstance(path, str) and len(path) > 0 and Path(path).exists():
                xs.append(series_tensor(path)); pm.append(1.0)
            else:
                xs.append(torch.zeros((SLICES_PER_SERIES, 1, IMG_SIZE, IMG_SIZE), dtype=torch.float32)); pm.append(0.0)
        x = torch.stack(xs, dim=0)
        plane_mask = torch.tensor(pm, dtype=torch.float32)
        y = torch.tensor([r.get(c, np.nan) for c in LABELS], dtype=torch.float32) if self.has_labels else torch.full((len(LABELS),), float("nan"))
        sw = torch.tensor(float(r.get("sample_w", 1.0)), dtype=torch.float32)
        return x, plane_mask, y, sw

class MILNet(nn.Module):
    def __init__(self, n_labels=12):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        feat_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.head = nn.Linear(feat_dim, n_labels)
    def forward(self, x, plane_mask):
        B, P, S, C, H, W = x.shape
        slice_valid = (x.abs().sum(dim=(3, 4, 5)) > 0).float()
        z = x.reshape(B * P * S, C, H, W)
        f = self.backbone(z).reshape(B, P, S, -1)
        w = slice_valid.unsqueeze(-1)
        series_feat = (f * w).sum(dim=2) / w.sum(dim=2).clamp_min(1.0)
        pm = plane_mask.unsqueeze(-1)
        study_feat = (series_feat * pm).sum(dim=1) / pm.sum(dim=1).clamp_min(1.0)
        return self.head(study_feat)

def weighted_masked_bce(logits, y, sw):
    m = ~torch.isnan(y)
    y2 = torch.nan_to_num(y, nan=0.0)
    loss = F.binary_cross_entropy_with_logits(logits, y2, reduction="none")  # B,12
    per = (loss * m).sum(dim=1) / m.sum(dim=1).clamp_min(1.0)              # B
    return (per * sw).sum() / sw.sum().clamp_min(1e-6)

def build_model(seed):
    torch.manual_seed(seed)
    model = MILNet(len(LABELS)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    return model, opt

print("model builder ready", flush=True)

In [ ]:
# Ячейка 5 — train v0.7 fixed: все GOLD в train, val только из non-gold

from pathlib import Path
from sklearn.metrics import roc_auc_score
import time, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# fallback-конфиг
if "SEEDS" not in globals(): SEEDS = [42, 1337]
if "N_TRAIN_STUDIES" not in globals(): N_TRAIN_STUDIES = 512
if "EPOCHS" not in globals(): EPOCHS = 2
if "BATCH_SIZE" not in globals(): BATCH_SIZE = 8
if "NUM_WORKERS" not in globals(): NUM_WORKERS = 0
if "VAL_FRAC" not in globals(): VAL_FRAC = 0.20
if "GRAD_CLIP" not in globals(): GRAD_CLIP = 1.0
if "LABELS" not in globals():
    LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

need = ["KneeStudyDS", "weighted_masked_bce", "build_model", "DEVICE", "USE_AMP"]
missing = [x for x in need if x not in globals()]
if missing:
    raise NameError(f"Не хватает объектов из Ячейки 4: {missing}. Сначала выполни Ячейку 4.")

if "train_df" not in globals():
    p = Path("/kaggle/working/train_selected_series_v0_7.csv")
    if p.exists():
        train_df = pd.read_csv(p)
        print("loaded train_df:", train_df.shape, flush=True)
    else:
        raise NameError("Нет train_df. Выполни Ячейки 1–3.")

if "sample_w" not in train_df.columns:
    train_df["sample_w"] = 1.0
    print("WARNING: sample_w не найден, все веса = 1.0", flush=True)

def fmt_eta(seconds):
    seconds = int(max(0, seconds))
    return f"{seconds // 60:02d}:{seconds % 60:02d}"

def quick_macro(logits_np, y_np):
    if len(logits_np) == 0:
        return np.nan
    prob = 1 / (1 + np.exp(-logits_np))
    aucs = []
    for j in range(len(LABELS)):
        m = ~np.isnan(y_np[:, j])
        if m.sum() > 0 and len(np.unique(y_np[m, j])) == 2:
            try:
                aucs.append(roc_auc_score(y_np[m, j], prob[m, j]))
            except Exception:
                pass
    return float(np.mean(aucs)) if aucs else np.nan

known_cnt = train_df[LABELS].notna().sum(axis=1)
gold_all = train_df[train_df["sample_w"] > 1].copy()
nongold_pool = train_df[(train_df["sample_w"] <= 1) & (known_cnt > 0)].copy()

print("gold_all:", len(gold_all), flush=True)
print("nongold with any weak label:", len(nongold_pool), flush=True)
if len(gold_all) == 0:
    print("WARNING: gold_all пуст. Проверь sample_w / GOLD_WEIGHT в Ячейке 3.", flush=True)

for seed in SEEDS:
    print("\n" + "#" * 90, flush=True)
    print("SEED", seed, flush=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    ng = nongold_pool.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    n_val = int(len(ng) * VAL_FRAC)
    va_df = ng.iloc[:n_val].copy()
    ng_train = ng.iloc[n_val:].copy()

    need_ng = max(0, N_TRAIN_STUDIES - len(gold_all))
    ng_train = ng_train.iloc[:need_ng].copy()

    tr_df = pd.concat([gold_all, ng_train], ignore_index=True)
    tr_df = tr_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    print("train/val sanity:", len(tr_df), len(va_df), flush=True)
    print("gold in train:", int((tr_df["sample_w"] > 1).sum()), "| gold in val:", int((va_df["sample_w"] > 1).sum()), flush=True)

    model, opt = build_model(seed)
    tr_loader = DataLoader(KneeStudyDS(tr_df, True), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    va_loader = DataLoader(KneeStudyDS(va_df, True), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS) if len(va_df) else None
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    for ep in range(EPOCHS):
        model.train()
        t0 = time.time()
        total_loss, total_n = 0.0, 0

        for bi, (x, pm, y, sw) in enumerate(tr_loader):
            x = x.to(DEVICE); pm = pm.to(DEVICE); y = y.to(DEVICE); sw = sw.to(DEVICE)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(x, pm)
                loss = weighted_masked_bce(logits, y, sw)

            opt.zero_grad(set_to_none=True)
            if USE_AMP:
                scaler.scale(loss).backward()
                if GRAD_CLIP:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                if GRAD_CLIP:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()

            total_loss += float(loss.item()) * len(y)
            total_n += len(y)

            if (bi % 10 == 0) or (bi == len(tr_loader) - 1):
                elapsed = time.time() - t0
                done = bi + 1
                eta = elapsed / max(done, 1) * (len(tr_loader) - done)
                print(f"[seed{seed}/ep{ep}] {done}/{len(tr_loader)} avg={total_loss / max(total_n, 1):.4f} elapsed={fmt_eta(elapsed)} eta={fmt_eta(eta)}", flush=True)

        va_auc = np.nan
        if va_loader is not None:
            model.eval()
            all_logits, all_y = [], []
            with torch.no_grad():
                for x, pm, y, sw in va_loader:
                    x = x.to(DEVICE); pm = pm.to(DEVICE)
                    with torch.cuda.amp.autocast(enabled=USE_AMP):
                        logits = model(x, pm)
                    all_logits.append(logits.float().cpu().numpy())
                    all_y.append(y.numpy())
            va_auc = quick_macro(np.vstack(all_logits), np.concatenate(all_y, axis=0)) if all_logits else np.nan

        print(f"[seed{seed}] epoch {ep} train_loss={total_loss / max(total_n, 1):.4f} sanity_val_macro={va_auc:.4f}", flush=True)

    out = f"/kaggle/working/milnet_v0_7_seed{seed}.pt"
    torch.save(model.state_dict(), out)
    print("saved", out, flush=True)

    del model, opt
    torch.cuda.empty_cache()

print("Ячейка 5 завершена.", flush=True)

In [ ]:
# Ячейка 6 — rank-average inference и submission.csv

if "SEEDS" not in globals(): SEEDS = [42, 1337]
if "BATCH_SIZE" not in globals(): BATCH_SIZE = 8
if "NUM_WORKERS" not in globals(): NUM_WORKERS = 0
if "LABELS" not in globals():
    LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

need = ["KneeStudyDS", "build_model", "DEVICE", "USE_AMP", "test_df", "sample_sub"]
missing = [x for x in need if x not in globals()]
if missing:
    raise NameError(f"Не хватает {missing}. Выполни предыдущие ячейки.")

def fmt_eta(seconds):
    seconds = int(max(0, seconds)); return f"{seconds // 60:02d}:{seconds % 60:02d}"

def rank_cols(arr):
    # arr: N,12 -> per-label ranks in [0,1]
    return pd.DataFrame(arr, columns=LABELS).rank(pct=True).to_numpy(np.float64)

test_loader = DataLoader(KneeStudyDS(test_df, False), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print("test batches:", len(test_loader), flush=True)

rank_preds = []
for seed in SEEDS:
    path = f"/kaggle/working/milnet_v0_7_seed{seed}.pt"
    if not Path(path).exists():
        print("missing model:", path, flush=True)
        continue
    model, _ = build_model(seed)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()

    preds = []
    t0 = time.time()
    with torch.no_grad():
        for bi, (x, pm, _, _) in enumerate(test_loader):
            x = x.to(DEVICE); pm = pm.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(x, pm)
            preds.append(torch.sigmoid(logits).float().cpu().numpy())
            if (bi % 20 == 0) or (bi == len(test_loader) - 1):
                elapsed = time.time() - t0; done = bi + 1
                eta = elapsed / max(done, 1) * (len(test_loader) - done)
                print(f"[infer seed{seed}] {done}/{len(test_loader)} elapsed={fmt_eta(elapsed)} eta={fmt_eta(eta)}", flush=True)

    p = np.vstack(preds) if preds else np.full((len(test_df), len(LABELS)), 0.5)
    np.save(f"/kaggle/working/preds_v0_7_seed{seed}.npy", p)
    rank_preds.append(rank_cols(p))
    print("seed done:", seed, flush=True)
    del model
    torch.cuda.empty_cache()

ens = np.mean(np.stack(rank_preds, axis=0), axis=0) if rank_preds else np.full((len(test_df), len(LABELS)), 0.5)

sub = sample_sub.copy().set_index("StudyInstanceUID")
pred_df = pd.DataFrame(ens, columns=LABELS)
pred_df.insert(0, "StudyInstanceUID", test_df["study"].values)
for _, r in pred_df.iterrows():
    if r["StudyInstanceUID"] in sub.index:
        sub.loc[r["StudyInstanceUID"], LABELS] = r[LABELS].values

sub = sub.reset_index()[["StudyInstanceUID"] + LABELS]
sub[LABELS] = sub[LABELS].fillna(0.5).clip(0.0, 1.0)
sub.to_csv("/kaggle/working/submission.csv", index=False)

print("saved: /kaggle/working/submission.csv", sub.shape, flush=True)
display(sub.head())
assert list(sub.columns) == list(sample_sub.columns)
assert sub[LABELS].isna().sum().sum() == 0
print("submission OK", flush=True)